# MNIST: specialists, generalist, and local Pareto exploration

Run the cells **from top to bottom**. Ten ResNet18 specialists learn one digit
versus the rest. A ResNet50 generalist sees two digits **overlaid on the same
28×28 canvas**. The first generalist run uses `mnist_okt`'s thresholded
teacher-label baseline. The second uses the ten unthresholded specialist
probabilities as ten competing distillation losses, then applies Frank–Wolfe
and MINRES tangent exploration.

The original executed notebook, including its old output, is preserved as
`mnist_okt_before_pareto.ipynb`. This notebook has no dependency on
`pareto_exploration.py` or `pareto_minres.py`.

Reference: Ma, Du, and Matusik, [*Efficient Continuous Pareto Exploration in
Multi-Task Learning*](https://arxiv.org/pdf/2006.16434), Equations (3)–(6),
Algorithm 1, and Section 5.

This is one local exploration: one seed, two chosen directions, both signs,
and sampled lines between parent and children. It does not implement the
paper's entire recursive exploration or recover the full ten-objective front.
The data split also differs from the old notebook, so old accuracy numbers
are not directly comparable. Each code cell begins with its purpose.

## Read this map first

`Original MNIST images → 10 specialists → soft teaching scores`

`Overlaid images → ResNet50 → 10 losses → Frank–Wolfe → MINRES → candidates → validation → test`

| Name | Meaning | Typical shape |
|---|---|---|
| `images` | A batch of overlaid grayscale images | `[B, 1, 28, 28]` |
| `targets` | True digit-presence flags for evaluation | `[B, 10]` |
| `teacher` / `q` | Fixed soft scores from the specialists | `[B, 10]` |
| `logits` / `z` | Raw student scores before sigmoid | `[B, 10]` |
| `losses` / `L` | Ten separate imitation errors; smaller is better | `[10]` |
| `theta` | The student parameters being differentiated | `[P]` when flattened |
| `G` | One parameter-gradient row per digit loss | `[10, P]` |
| `alpha` | Nonnegative gradient-balancing weights summing to one | `[10]` |
| `beta` | Coefficients choosing a tangent system's right-hand side | `[10]` |
| `v` | A direction for changing selected model parameters | `[P]` |

`B` means batch size and `P` means the number of selected parameters.
Training updates parameters; evaluation measures a frozen model. The network
has ten independent sigmoid outputs because multiple digits may be present.
The **Pareto set** contains model parameters; the **Pareto front** contains
their ten loss values. Several sampled points may be non-dominated even when
none is a true optimum. Read the diagnostics before interpreting the plot.


## 0. Requirements and imports

Use a Python environment with `torch`, `torchvision`, `numpy`, and
`matplotlib`. MNIST downloads automatically on the first run. A CUDA GPU is
strongly recommended; training eleven ResNets and taking second derivatives
can take a long time. If packages are missing, install them in your notebook
environment before running all cells.

In [ ]:
# Load numerical tools, neural-network components, data loaders, and plotting.
# PyTorch handles differentiation; torchvision supplies MNIST and ResNets.

import copy
import json
import random
from dataclasses import dataclass
from math import hypot, sqrt, isfinite
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

## 1. Settings

Start with the values below for the full experiment. If you only want a quick
check, reduce `SPEC_TRAIN_SIZE`, `STACK_TRAIN_SAMPLES`, and the epoch counts.
The Pareto section is a **local approximation**, not a proof that every point
is globally Pareto optimal.

In [ ]:
# Training budgets: an epoch is one complete pass over the chosen training pairs.
SEED = 42
BATCH_SIZE = 128
SPEC_EPOCHS = 3
SPEC_TRAIN_SIZE = 20_000
GEN_EPOCHS = 7
STACK_TRAIN_SAMPLES = 30_000
STACK_VAL_SAMPLES = 2_000
STACK_TEST_SAMPLES = 5_000

# Exploration budgets: these short runs demonstrate the method; inspect residuals.
SOFT_SEED_EPOCHS = 2
CORRECTOR_EPOCHS = 1
PARETO_BETA_PAIRS = [(7, 9), (3, 8)]
TANGENT_BATCH_SIZE = 16
MINRES_MAX_ITER = 10
MINRES_TOLERANCE = 1e-3
RELATIVE_TANGENT_STEP = 0.01
PREFERENCE_STRENGTH = 0.5
INTERPOLATION_FRACTIONS = [0.5]

# Use an available accelerator; seeds make data sampling repeatable.
DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
# Checkpoints and the experiment report are saved here relative to the notebook.
ARTIFACT_DIR = Path('artifacts')
ARTIFACT_DIR.mkdir(exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print('Device:', DEVICE)

## 2. MNIST and a separate validation split

Specialists and the generalist train on `base_train`. Model selection and
Pareto pruning use `base_val`; the final `test_full` remains untouched until
the end. Each pair dataset samples fixed combinations from its base split.

In [ ]:
# Normalize each grayscale image, then split ORIGINAL images before forming pairs.
# This prevents a validation digit image from also appearing in training pairs.

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
train_full = datasets.MNIST('./data', train=True, download=True, transform=transform)
# Loading the held-out dataset is fine; model selection never reads its results.
test_full = datasets.MNIST('./data', train=False, download=True, transform=transform)
split_indices = np.random.RandomState(SEED).permutation(len(train_full))
base_train = Subset(train_full, split_indices[:55_000].tolist())
base_val = Subset(train_full, split_indices[55_000:].tolist())
print('Original train:', len(train_full), '| training base:', len(base_train),
      '| validation base:', len(base_val), '| final test:', len(test_full))

## 3. Models and binary specialist data

Specialist `d` sees a positive label for digit `d` and a negative label for
every other digit. The ResNets start with random weights and are adapted to
one-channel 28×28 MNIST input.

Convolutions learn image patterns. A residual block learns a change to its
input and adds that change back through a skip connection. The final linear
layer turns the learned features into logits. ResNet18 and ResNet50 have
different depths; each specialist has its own independent parameters.

In [ ]:
# A specialist answers one yes/no question; the generalist answers ten at once.
# conv1 accepts one channel; fc sets the output count. Remaining layers are ResNet.

def make_resnet18(num_outputs=2):
    # Specialist: ResNet18 with a 2-output head -> [not_d, d]
    model = models.resnet18(weights=None)   # weights=None -> start from random weights
    model.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
    # Return raw logits. The loss function applies the required probability transform.
    model.fc = nn.Linear(model.fc.in_features, num_outputs)
    return model

def make_resnet50(num_outputs=10):
    # Generalist: ResNet50 with a 10-output head -> one output per digit 0..9
    model = models.resnet50(weights=None)
    model.conv1 = nn.Conv2d(1, 64, kernel_size=3, stride=1, padding=1, bias=False)
    # Return raw logits. The loss function applies the required probability transform.
    model.fc = nn.Linear(model.fc.in_features, num_outputs)
    return model

In [ ]:
# Reuse each image with a different binary label for each specialist.
# The training subset controls cost and never draws from validation images.

class BinaryMNIST(Dataset):
    # Wraps MNIST and changes the label to: 1 if label == digit else 0
    def __init__(self, base_dataset, digit):
        self.base = base_dataset
        self.digit = digit

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        img, label = self.base[idx]
        return img, int(label == self.digit)


if SPEC_TRAIN_SIZE is None:
    spec_train_base = base_train
else:
    sample_indices = np.random.permutation(len(base_train))[:SPEC_TRAIN_SIZE]
    spec_train_base = Subset(base_train, sample_indices.tolist())

## 4. Train and validate the ten specialists

The positive class gets weight 9 because one digit is much rarer than all
other digits combined. Training each model uses its own optimizer.

A training batch goes through five steps: clear old gradients, predict,
measure loss, differentiate the loss, and update parameters. Adam uses those
gradients with adaptive step sizes. `model.train()` lets BatchNorm update its
running statistics; `model.eval()` freezes those statistics.
`torch.no_grad()` separately disables gradient recording for evaluation.

In [ ]:
# Train one teacher: image -> two logits -> weighted cross-entropy -> gradient update.

def train_specialist(digit):
    model = make_resnet18(num_outputs=2).to(DEVICE)

    train_loader = DataLoader(BinaryMNIST(spec_train_base, digit),
                              batch_size=BATCH_SIZE, shuffle=True)

    # class 0 = "not d" (weight 1), class 1 = "d" (weight 9 because it is ~9x rarer)
    class_weights = torch.tensor([1.0, 9.0]).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    for epoch in range(SPEC_EPOCHS):
        model.train()
        loss_numerator = weight_denominator = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            optimizer.zero_grad()              # 1) clear old gradients
            outputs = model(images)            # 2) forward pass
            loss = criterion(outputs, labels)  # 3) how wrong are we?
            loss.backward()                    # 4) compute gradients
            optimizer.step()                   # 5) update weights

            # Weighted CE divides by the sum of sample weights, not the batch size.
            batch_weight = class_weights[labels].sum().item()
            loss_numerator += loss.item() * batch_weight
            weight_denominator += batch_weight
        print(f"  specialist {digit} | epoch {epoch+1}/{SPEC_EPOCHS} | loss {loss_numerator/weight_denominator:.4f}")

    model.eval()
    torch.save({key: value.detach().cpu() for key, value in model.state_dict().items()}, ARTIFACT_DIR / f"specialist_{digit}.pt")
    return model

In [ ]:
# Train the ten independent teachers, then freeze them for all student experiments.
# Frozen teachers give every candidate the same teaching signal.
specialists = []
for digit in range(10):
    specialist = train_specialist(digit)
    specialist.eval()
    specialist.requires_grad_(False)
    specialists.append(specialist)

In [ ]:
# Evaluate without gradient recording or BatchNorm updates.
# Accuracy includes negatives; recall measures how many real digit-d images are found.

@torch.no_grad()
def evaluate_specialist(model, digit, base_dataset):
    model.eval()
    loader = DataLoader(BinaryMNIST(base_dataset, digit), batch_size=256)
    correct, total, pos_correct, pos_total = 0, 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.numel()
        pos_mask = labels == 1
        pos_correct += (preds[pos_mask] == 1).sum().item()
        pos_total += pos_mask.sum().item()
    return correct / total, pos_correct / max(pos_total, 1)

In [ ]:
# Check every teacher before using its outputs as student training targets.

for digit, specialist in enumerate(specialists):
    accuracy, recall = evaluate_specialist(specialist, digit, base_val)
    print(f'Specialist {digit}: validation accuracy {accuracy:.3f}, recall {recall:.3f}')

In [ ]:
# Combine the positive-class scores of all teachers into a [batch, 10] matrix.
# Each teacher uses a two-class softmax. Across teachers, scores need not sum to 1.

@torch.no_grad()
def specialists_predict(images):
    # images: (B, 1, 28, 28) -> returns (B, 10): probability that each digit is present
    probs = []
    for m in specialists:
        m.eval()
        p = F.softmax(m(images), dim=1)[:, 1]   # index 1 = "it is digit d"
        probs.append(p)
    return torch.stack(probs, dim=1)

In [ ]:
# Single-digit sanity check: pick the largest teacher score for each original image.
# This argmax check is only for single digits; the generalist can output multiple digits.

validation_single_loader = DataLoader(base_val, batch_size=256)
correct = 0
for images, labels in validation_single_loader:
    images, labels = images.to(DEVICE), labels.to(DEVICE)
    correct += (specialists_predict(images).argmax(dim=1) == labels).sum().item()
print(f'Specialist team validation accuracy: {correct / len(base_val):.3f}')

## 5. Build overlaid two-digit images

`StackedMNIST` returns five items: the overlay, its true multi-hot label, the
two digit IDs, and the two original images. The generalist sees only the
overlay. Teachers see each original image separately; **do not split the
overlay into left and right halves**.

Pairs are sampled with replacement. The two digits may be equal.
A pair `(7, 7)` has one active label for 7: the task predicts which digits
are present, not how many times each appears.

In [ ]:
# Fix pair indices once, so different models are evaluated on identical overlays.

class StackedMNIST(Dataset):
    def __init__(self, base_dataset, n_samples, seed):
        self.base = base_dataset
        rng = np.random.RandomState(seed)
        # Pick which two MNIST images to overlay for every sample (fixed once, so it is repeatable)
        self.a_idx  = rng.randint(0, len(base_dataset), n_samples)
        self.b_idx = rng.randint(0, len(base_dataset), n_samples)

    def __len__(self):
        return len(self.a_idx)

    def __getitem__(self, i):
        img_a, lab_a = self.base[int(self.a_idx[i])]
        img_b, lab_b = self.base[int(self.b_idx[i])]

        # Superimpose the two digits on the same canvas -> (1, 28, 28)
        # max keeps both strokes at full brightness where they cross
        stacked = torch.maximum(img_a, img_b)

        # Multi-hot means one independent presence flag per digit.
        # If both originals have the same label, that flag is still 1.
        target = torch.zeros(10)                     # multi-hot label
        target[lab_a] = 1.0
        target[lab_b] = 1.0
        return stacked, target, torch.tensor([lab_a, lab_b]), img_a, img_b

In [ ]:
# Create separate fixed pair datasets. Shuffle training order only.
# Validation selects models; test data is reserved for the final evaluation.

stack_train = StackedMNIST(base_train, STACK_TRAIN_SAMPLES, seed=1)
stack_val = StackedMNIST(base_val, STACK_VAL_SAMPLES, seed=2)
stack_test = StackedMNIST(test_full, STACK_TEST_SAMPLES, seed=3)
train_loader = DataLoader(stack_train, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(stack_val, batch_size=BATCH_SIZE)
test_loader = DataLoader(stack_test, batch_size=BATCH_SIZE)
print('Pairs:', len(stack_train), 'train,', len(stack_val), 'validation,', len(stack_test), 'test')

In [ ]:
# Visually verify that overlays and their true presence labels agree.

fig, axes = plt.subplots(1, 4, figsize=(9, 2.5))
for i, ax in enumerate(axes):
    image, target, digits, _, _ = stack_train[i]
    ax.imshow(image.squeeze().numpy(), cmap='gray')
    ax.set_title(f'{digits.tolist()} → {torch.where(target == 1)[0].tolist()}')
    ax.axis('off')
plt.show()

## 6. Specialist teaching signals

The ten specialists make probability vectors for both original digits. The
pair target is `max` of those vectors, one value per digit. Keeping those
values unthresholded defines the **soft-label** Pareto objectives later.

For one overlay containing originals `a` and `b`, specialist `d` outputs
`p_d(a)` and `p_d(b)`. We set `q_d = max(p_d(a), p_d(b))`. If its confidence is
0.73, the teaching signal stays 0.73; turning it into 1 would discard
information. The generalist emits ten **logits** `z_d`, and the ten losses are

`L_d(θ) = mean_batch BCEWithLogits(z_d(θ), q_d)`, for `d = 0,…,9`.

`θ` means the generalist's trainable parameters. Every `L_d` is a separate
objective. One model must satisfy all ten, so improving one digit can worsen
another through the shared ResNet features. True digit labels are used later
to judge recognition; they do not define these distillation objectives.

For one sample, the probability of digit `d` is `s_d = sigmoid(z_d)` and its loss is

$$\ell_d=-q_d\log s_d-(1-q_d)\log(1-s_d),\qquad
L_d=\frac{1}{B}\sum_{b=1}^{B}\ell_{b,d}.$$

`BCEWithLogits` evaluates this stably from logits. Its derivative with respect
to a logit is `(s_d - q_d) / B`, so it pushes the student's score toward the
teacher's score. `max` is this experiment's chosen aggregation rule; it is not
an exact probability formula for a union of events. Class-weighted specialists
also need not produce calibrated probabilities. These are teaching scores.

In [ ]:
# Build SOFT targets, then keep the ten loss columns separate for multi-objective work.

@torch.no_grad()
def soft_teacher_targets(img_a, img_b):
    """Ten unthresholded specialist hints for the two original MNIST images.

    The generalist sees their overlaid image, but the specialists see each
    original 28x28 image. Keeping probabilities here is essential for the
    soft-label distillation objectives in the tangent experiment.
    """
    return torch.maximum(specialists_predict(img_a.to(DEVICE)),
                         specialists_predict(img_b.to(DEVICE)))

def per_digit_distillation_losses(logits, teacher_probs):
    """L_d = mean over the batch of BCE(logit_d, teacher_probability_d)."""
    if logits.shape != teacher_probs.shape or logits.ndim != 2 or logits.shape[1] != 10:
        raise ValueError("Expected matching [batch, 10] logits and teacher probabilities")
    # reduction="none": [B, 10]. Mean over samples only: [10].
    # Averaging over both dimensions here would erase the separate objectives.
    return F.binary_cross_entropy_with_logits(
        logits, teacher_probs, reduction="none").mean(dim=0)

## 7. Train the `mnist_okt` thresholded-teacher baseline

For this baseline only, probabilities at least 0.5 become label 1. The model
trains on teacher decisions rather than true pair labels. Unlike the old
notebook, the optimizer is created for the model passed to this function.

In [ ]:
# Train a student against fixed teacher targets using one equal-weight scalar loss.
def train_generalist(model, loader, epochs, target_mode, lr=1e-3,
                     save_file=None, freeze_batchnorm=False):
    if target_mode not in {'binary', 'soft'}:
        raise ValueError('target_mode must be binary or soft')
    # A fresh optimizer must own the parameters of THIS model.
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for epoch in range(epochs):
        # eval() freezes BatchNorm statistics; it does NOT disable differentiation.
        # Soft exploration uses the same fixed statistics for training and Hessians.
        model.eval() if freeze_batchnorm else model.train()
        total_loss = total_samples = 0
        for images, _, _, img_a, img_b in loader:
            images = images.to(DEVICE)
            teacher = soft_teacher_targets(img_a, img_b)
            if target_mode == 'binary':
                teacher = (teacher >= 0.5).float()  # baseline only
            logits = model(images)
            losses = per_digit_distillation_losses(logits, teacher)
            loss = losses.mean()  # (L_0 + ... + L_9) / 10
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            # Weight by sample count so a short final batch is counted correctly.
            total_loss += loss.item() * images.size(0)
            total_samples += images.size(0)
        print(f'{target_mode} epoch {epoch + 1}/{epochs}: teacher BCE {total_loss / total_samples:.4f}')
    model.eval()
    if save_file is not None:
        torch.save({key: value.detach().cpu() for key, value in model.state_dict().items()},
                   ARTIFACT_DIR / save_file)
    return model

In [ ]:
# Establish the thresholded-teacher baseline before starting the soft-label experiment.

generalist = make_resnet50(num_outputs=10).to(DEVICE)
generalist = train_generalist(generalist, train_loader, GEN_EPOCHS,
                             target_mode='binary', save_file='generalist_binary_teacher.pt')

## 8. Compare decision rules on validation data

The fixed 0.5 rule is the main metric. The gap, relative-max, and normalized
relative rules reproduce the extra decision experiments from `mnist_okt`.
These alternatives are diagnostic comparisons. All later metrics keep the
fixed 0.5 rule, so comparisons are consistent. Pareto objectives always use
the unthresholded soft teacher targets. Per-digit accuracy can look high
by predicting many absent digits; exact match requires the entire set to be right.

In [ ]:
# Convert ten independent presence scores into predicted digit sets.
# These rules affect reported predictions, never the differentiable training losses.

@torch.no_grad()
def prediction_flags(probabilities, rule='fixed', ratio=0.5):
    if rule == 'fixed':
        return (probabilities > 0.5).float()
    # Gap is a heuristic: it always chooses 1..9 labels, even if every score is low.
    if rule == 'gap':
        sorted_p, indices = probabilities.sort(dim=1, descending=True)
        k = (sorted_p[:, :-1] - sorted_p[:, 1:]).argmax(dim=1) + 1
        rank = torch.arange(10, device=probabilities.device).unsqueeze(0)
        return torch.zeros_like(probabilities).scatter(1, indices,
                 (rank < k.unsqueeze(1)).float())
    if rule == 'relative':
        return (probabilities > ratio * probabilities.max(dim=1, keepdim=True).values).float()
    if rule == 'normalized_relative':
        minimum = probabilities.min(dim=1, keepdim=True).values
        maximum = probabilities.max(dim=1, keepdim=True).values
        normalized = (probabilities - minimum) / (maximum - minimum + 1e-8)
        return (normalized > ratio).float()
    raise ValueError(rule)


@torch.no_grad()
def evaluate_decisions(model, loader, rule='fixed', ratio=0.5):
    model.eval()
    exact = total = 0
    correct = torch.zeros(10)
    for images, targets, *_ in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        probabilities = torch.sigmoid(model(images))
        predicted = prediction_flags(probabilities, rule, ratio)
        exact += (predicted == targets).all(dim=1).sum().item()
        correct += (predicted == targets).float().sum(dim=0).cpu()
        total += targets.size(0)
    # Exact match: every one of the ten presence flags must match the true target.
    return {'exact_match': exact / total, 'per_digit_accuracy': (correct / total).tolist()}


for rule, ratio in [('fixed', 0.5), ('gap', 0.5),
                    ('relative', 0.5), ('normalized_relative', 0.1)]:
    metrics = evaluate_decisions(generalist, val_loader, rule, ratio)
    print(f'{rule:20} validation exact match: {metrics["exact_match"]:.3f}')

In [ ]:
# Show qualitative examples from validation data alongside their true digit sets.

@torch.no_grad()
def show_predictions(model, dataset=stack_val, count=6):
    model.eval()
    fig, axes = plt.subplots(1, count, figsize=(2.4 * count, 3))
    model_device = next(model.parameters()).device
    for ax in np.atleast_1d(axes):
        image, _, digits, _, _ = dataset[random.randrange(len(dataset))]
        probabilities = torch.sigmoid(model(image.unsqueeze(0).to(model_device)))[0].cpu()
        predicted = torch.where(probabilities > 0.5)[0].tolist()
        truth = sorted(set(digits.tolist()))
        ax.imshow(image.squeeze().numpy(), cmap='gray')
        ax.set_title(f'true {truth}\npred {predicted}',
                     color='green' if predicted == truth else 'red')
        ax.axis('off')
    plt.show()


show_predictions(generalist)

## 9. Create a seed for ten soft-label objectives

Each objective `L_d` is the batch mean BCE between the student's digit-`d`
logit and specialist probability `q_d`. This experiment fine-tunes a copy of
the baseline using those soft probabilities. The true labels are retained
for external evaluation, not used in these ten distillation losses.

The seed learns with equal loss weights. During this phase and all later
corrections, BatchNorm running statistics are frozen. Otherwise the model
state would change with each batch and the Hessian objective would differ
from the objective used for correction. Network weights still receive gradients.

In [ ]:
# Continue from the baseline, replacing hard teacher labels with soft scores.
# This creates the common starting checkpoint for every tangent experiment.

soft_seed = copy.deepcopy(generalist)
generalist.cpu()  # release the baseline from accelerator memory
soft_seed = train_generalist(soft_seed, train_loader, SOFT_SEED_EPOCHS,
                             target_mode='soft', lr=1e-4,
                             save_file='generalist_soft_seed.pt', freeze_batchnorm=True)
print('Soft seed validation:', evaluate_decisions(soft_seed, val_loader))
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

## 10. Frank–Wolfe: find a balance among ten task gradients

At a fixed model, let `g_d = gradient_theta L_d`. Frank–Wolfe solves
`min_alpha ||sum_d alpha_d g_d||²` on the simplex (`alpha >= 0`, sum 1).
The `10×10` Gram matrix `G Gᵀ` is enough. In the tangent function below,
the paper's equation uses unnormalized parameter gradients and `lam=0`.

Think of each `g_d` as the direction in parameter space that would increase
digit `d`'s loss fastest. A training step in the opposite direction lowers
that loss locally. The weighted sum `c = Σ alpha_d g_d` is a compromise across
digits. Frank–Wolfe starts with equal weights. At each iteration it picks the
simplex corner that most decreases `||c||²`, then moves partway toward it.

The weights are constrained to be nonnegative and sum to one. If the smallest
possible `||c||` is near zero, the model is near a **first-order Pareto
stationary** point in the selected parameter subspace. This condition alone
does not guarantee global Pareto optimality. In code, `G` has shape
`[10, number_of_selected_parameters]`, while `G @ G.T` is just `10×10`.

Your task 3 is the solver below. Writing `M = G Gᵀ` gives
`f(alpha) = alphaᵀ M alpha`. Its gradient is `2 M alpha`. The algorithm picks
the smallest component of that gradient, giving a corner `e_j` of the simplex.
It then uses `alpha_new = (1-gamma) alpha + gamma e_j`. Since `gamma` stays in
`[0,1]`, the weights stay nonnegative and sum to one. This solver chooses
weights at a fixed model; the later correction updates the model itself.

In [ ]:
# Solve the small task-weight problem using Frank-Wolfe and exact quadratic line search.
@torch.no_grad()
def frank_wolfe_simplex(M, pref=None, lam=0.0, iters=20, tol=1e-6):
    """Minimize alpha.T @ M @ alpha + lam * ||alpha-pref||^2 on the simplex.

    M is a gradient Gram matrix. The tangent calculation uses lam=0.
    A preference penalty changes this weight problem; it does not itself
    generate or certify a Pareto front.
    """
    if M.ndim != 2 or M.shape[0] != M.shape[1] or M.shape[0] == 0:
        raise ValueError('M must be a nonempty square Gram matrix')
    if not M.is_floating_point() or not torch.isfinite(M).all():
        raise ValueError('M must contain finite floating-point entries')
    if lam < 0 or iters < 1 or tol <= 0:
        raise ValueError('Require lam >= 0, iters >= 1, tol > 0')
    K = M.shape[0]
    # Match dtype as well as device: this also supports float64 numerical checks.
    alpha = M.new_full((K,), 1.0 / K)
    if pref is None:
        pref, lam = alpha.clone(), 0.0
    pref = torch.as_tensor(pref, device=M.device, dtype=M.dtype)
    if (pref.shape != (K,) or not torch.isfinite(pref).all()
            or (pref < 0).any() or not torch.isclose(pref.sum(), M.new_tensor(1.0))):
        raise ValueError('pref must be a finite probability vector')
    # This Hessian is only K x K. It is NOT the model-parameter Hessian used by MINRES.
    H = 2 * M + 2 * lam * torch.eye(K, device=M.device, dtype=M.dtype)
    for _ in range(iters):
        grad = H @ alpha - 2 * lam * pref
        j = torch.argmin(grad)           # best simplex corner for the linearized objective
        d = -alpha.clone()
        d[j] += 1.0                     # d = e_j - alpha
        gap = -(grad @ d)               # available decrease in the linear approximation
        if gap <= tol:
            break
        curvature = d @ H @ d
        # A flat segment is minimized at its endpoint; otherwise use exact line search.
        gamma = (gap / curvature).clamp(0, 1) if curvature > 0 else M.new_tensor(1.0)
        alpha = alpha + gamma * d
    return alpha


def _selftest():
    """Sanity check: Frank-Wolfe must match the closed-form 2-task MGDA solution."""
    g1, g2 = torch.tensor([1.0, 0.2]), torch.tensor([0.1, 1.0])
    G = torch.stack([g1, g2])
    alpha = frank_wolfe_simplex(G @ G.T, iters=200, tol=1e-10)
    closed = ((g2 @ (g2 - g1)) / ((g1 - g2) @ (g1 - g2))).clamp(0, 1)   # alpha for g1
    assert abs(alpha[0].item() - closed.item()) < 1e-3, (alpha, closed)
    assert abs(alpha.sum().item() - 1) < 1e-5 and (alpha >= 0).all()
    print("FW self-test passed:", alpha.tolist())

_selftest()

## 11. Matrix-free MINRES

The weighted Hessian has one row per model parameter and is far too large
to construct. MINRES receives a function that calculates `H @ v` using
automatic differentiation. The solver below stores only a few parameter
vectors, handles symmetric indefinite systems, and reports its measured
relative residual.

Here the Hessian `H` describes how the weighted gradient `c` changes when
model parameters move a little. `H @ v` is a Hessian-vector product, computed
by differentiating a gradient-vector product. The full Hessian is never saved.
MINRES finds a vector `v` whose product `H @ v` approximates a requested
right-hand side `b`. A small relative residual
`||H @ v - b|| / ||b||` indicates that this linear solve worked on the chosen
batch. It does not measure model accuracy or Pareto optimality.

In [ ]:
# Solve A @ x = b using products with A, without storing the full matrix.
# The alpha/beta scalars INSIDE this solver are recurrence coefficients,
# unrelated to task weights alpha or the ten-entry exploration beta.

@dataclass
class MinresResult:
    solution: object
    iterations: int
    relative_residual: float
    converged: bool

def _norm(vector):
    squared = float(vector @ vector)
    if not isfinite(squared):
        raise ValueError('MINRES encountered a non-finite vector')
    return sqrt(max(squared, 0.0))

def minres(matvec, b, max_iter=50, tolerance=1e-4):
    """Approximately solve A x = b for symmetric (possibly indefinite) A.

    The stopping test uses the MINRES residual estimate. The returned residual
    is measured directly with one final matrix-vector product, so callers can
    tell whether a truncated solve is accurate enough for their experiment.
    """
    if max_iter < 1 or tolerance <= 0:
        raise ValueError("max_iter must be positive and tolerance must be > 0")
    x = b * 0
    beta_initial = _norm(b)
    if beta_initial == 0:
        return MinresResult(x, 0, 0.0, True)

    # Lanczos three-term recurrence, followed by Givens rotations of its
    # tridiagonal least-squares system. Only a few vectors of b's size live.
    r_previous = b
    r_current = b
    y = b
    beta_previous = 0.0
    beta = beta_initial
    dbar = epsln = 0.0
    cs, sn = -1.0, 0.0
    phibar = beta_initial
    w_previous = b * 0
    w_current = b * 0
    iterations = 0

    for iteration in range(1, max_iter + 1):
        # Expand a small search space using a new matrix-vector product.
        v = y / beta
        y = matvec(v)
        if iteration > 1:
            y = y - (beta / beta_previous) * r_previous
        alpha = float(v @ y)
        y = y - (alpha / beta) * r_current
        r_previous, r_current = r_current, y
        beta_previous, beta = beta, _norm(r_current)

        # Givens rotations solve the small least-squares problem within that space.
        old_epsln = epsln
        delta = cs * dbar + sn * alpha
        gbar = sn * dbar - cs * alpha
        epsln = sn * beta
        dbar = -cs * beta
        gamma = hypot(gbar, beta)
        if gamma == 0:
            break
        cs, sn = gbar / gamma, beta / gamma
        phi = cs * phibar
        phibar = sn * phibar

        # Update the full solution using only a few stored vectors.
        w_old = w_previous
        w_previous = w_current
        w_current = (v - old_epsln * w_old - delta * w_previous) / gamma
        x = x + phi * w_current
        iterations = iteration
        if abs(phibar) <= tolerance * beta_initial or beta == 0:
            break

    # Measure the actual residual; a short run may stop before it is small.
    residual = _norm(b - matvec(x)) / beta_initial
    return MinresResult(x, iterations, residual, residual <= tolerance)

In [ ]:
# A tiny system with one positive and one negative eigenvalue has a known answer.
# This checks solver arithmetic before using an implicit neural-network Hessian.

matrix = torch.tensor([[2.0, 0.0], [0.0, -1.0]])
rhs = torch.tensor([2.0, 1.0])
solver_check = minres(lambda vector: matrix @ vector, rhs, max_iter=5, tolerance=1e-6)
assert solver_check.converged
assert torch.allclose(solver_check.solution, torch.tensor([1.0, -1.0]), atol=1e-5)
print('MINRES check:', solver_check.iterations, 'iterations, relative residual',
      solver_check.relative_residual)

## 12. Tangent predictor: from task gradients to a parameter direction

Your task 4 uses the ten soft losses from Section 6. At the fixed seed, define
`g_d = gradient_theta L_d`, find `alpha` with Frank–Wolfe, and set
`c = sum_d alpha_d g_d`. The linear system is

$$H v = \sum_d\beta_d(g_d-c),\qquad H=\sum_d\alpha_d\nabla^2 L_d.$$

Here `alpha` is held constant while differentiating the weighted loss. `beta`
chooses a combination of corrected gradients on the right-hand side; `v`
contains one entry per selected model parameter. The Hessian is never built.
For `beta = e7-e9`, the entries sum to zero and the right-hand side is `g_7-g_9`.
**A positive beta entry does not promise that this digit's loss will decrease.**
We predict each loss change using `delta L_d ≈ step_size * g_dᵀ v` and try both signs.

For a concrete sign check, let `L_1=(theta-1)^2` and `L_2=(theta+1)^2`.
At `theta=0`, `alpha=(0.5,0.5)`, `H=2`, and `beta=(1,-1)` gives `v=-2`.
A positive step along `v` improves **L_2** and worsens **L_1**.

The default selects the last shared ResNet block and the final head. This is
a parameter-subspace approximation, and a 16-pair calibration batch is a
sample estimate. The printed stationarity ratio applies to that batch and
subspace only. The 0.1 warning threshold is a heuristic, not a theorem.
A high ratio may reflect inadequate training or batch noise. The corrected
gradient equation accommodates an imperfect seed, but cannot certify an optimum.

In [ ]:
# Measure task geometry once per model: losses -> gradients -> Gram matrix -> alpha.
def task_gradient_geometry(losses, params):
    sizes = [p.numel() for p in params]
    # Offload task rows to CPU; ten full parameter vectors can be large on a GPU.
    G = torch.empty((len(losses), sum(sizes)), dtype=params[0].dtype, device='cpu')
    for task, loss in enumerate(losses):
        grads = torch.autograd.grad(loss, params, retain_graph=True, allow_unused=True)
        offset = 0
        for grad, size in zip(grads, sizes):
            G[task, offset:offset + size] = 0 if grad is None else grad.detach().reshape(-1).cpu()
            offset += size
    norms = G.norm(dim=1)
    # Scale ALL gradients by the same number. Per-task normalization would change Eq. (3).
    scale = norms.max().clamp_min(1e-12)
    gram = (G @ G.T) / scale.square()
    alpha = frank_wolfe_simplex(gram, lam=0.0, iters=200, tol=1e-8)
    correction = alpha @ G
    # A negative cosine indicates gradients that disagree on an immediate descent direction.
    unit_rows = G / norms.clamp_min(1e-12).unsqueeze(1)
    summary = {
        'alpha': alpha.tolist(),
        'stationarity_ratio': (correction.norm() / norms.mean().clamp_min(1e-12)).item(),
        'fw_gap': max(0.0, float(2 * (alpha @ gram @ alpha - (gram @ alpha).min()))),
        'gradient_norms': norms.tolist(),
        'gradient_cosines': (unit_rows @ unit_rows.T).tolist(),
    }
    return G, alpha, correction, summary


def measure_stationarity(model, batch, params):
    """Diagnostic on the selected batch/subspace; not a full-data optimality certificate."""
    was_training = model.training
    model.eval()
    try:
        images, _, _, img_a, img_b = batch
        device = next(model.parameters()).device
        losses = per_digit_distillation_losses(
            model(images.to(device)), soft_teacher_targets(img_a, img_b).to(device))
        _, _, _, summary = task_gradient_geometry(losses, list(params))
        return summary
    finally:
        model.train(was_training)


# Convert the task combination beta into a parameter direction using Hessian-vector products.
def minres_teacher_tangent(model, batch, beta, params=None, max_iter=50,
                           tolerance=1e-4, damping=0.0):
    """Solve the corrected tangent system and return a unit-length direction plus diagnostics."""
    if damping < 0:
        raise ValueError('damping must be nonnegative')
    params = [p for p in (model.parameters() if params is None else params) if p.requires_grad]
    if not params:
        raise ValueError('No trainable parameters were selected')
    device, dtype = params[0].device, params[0].dtype
    if any(p.device != device or p.dtype != dtype for p in params):
        raise ValueError('Selected parameters must share a device and dtype')
    beta = torch.as_tensor(beta, dtype=dtype, device='cpu')
    if beta.shape != (10,) or not torch.isfinite(beta).all():
        raise ValueError('beta must be a finite length-10 vector')
    images, _, _, img_a, img_b = batch
    was_training = model.training
    model.eval()  # keep BatchNorm statistics fixed throughout the linear solve
    try:
        soft = soft_teacher_targets(img_a, img_b).to(device)
        losses = per_digit_distillation_losses(model(images.to(device)), soft)
        G, alpha, correction, diagnostics = task_gradient_geometry(losses, params)
        rhs_cpu = beta @ G - beta.sum() * correction
        if rhs_cpu.norm() <= 1e-12:
            raise ValueError('The chosen beta has no tangent signal for these tasks')

        # Differentiate the weighted gradient while treating alpha as a fixed coefficient.
        sizes = [p.numel() for p in params]
        weighted_loss = (alpha.to(device) * losses).sum()
        first_grads = torch.autograd.grad(weighted_loss, params, create_graph=True,
                                          retain_graph=True, allow_unused=True)

        def hessian_vector_product(vector):
            # Reassemble the flat vector into parameter-shaped pieces, then compute H @ vector.
            parts = [chunk.view_as(p) for chunk, p in zip(vector.split(sizes), params)]
            active = [(grad, part) for grad, part in zip(first_grads, parts)
                      if grad is not None and grad.requires_grad]
            second = torch.autograd.grad(
                [item[0] for item in active], params,
                grad_outputs=[item[1] for item in active],
                retain_graph=True, allow_unused=True) if active else [None] * len(params)
            flat = torch.cat([
                (grad.detach() if grad is not None else torch.zeros_like(p)).reshape(-1)
                for p, grad in zip(params, second)])
            # damping=0 matches the stated system; positive damping solves (H + damping I) v = rhs.
            return flat + damping * vector

        solution = minres(hessian_vector_product, rhs_cpu.to(device),
                          max_iter=max_iter, tolerance=tolerance)
        length = solution.solution.norm()
        if not torch.isfinite(length) or length <= 1e-12:
            raise ValueError('MINRES returned a zero or non-finite direction')
        direction = (solution.solution / length).detach()
        return {
            'direction': direction,
            'beta': beta,
            'teacher_losses': losses.detach().cpu(),
            'predicted_slopes': (G @ direction.cpu()).tolist(),
            **diagnostics,
            'minres_iterations': solution.iterations,
            'minres_relative_residual': solution.relative_residual,
            'minres_converged': solution.converged,
            'parameter_count': sum(sizes),
        }
    finally:
        model.train(was_training)


@torch.no_grad()
def apply_tangent_step(params, direction, step_size):
    """Move a candidate's selected parameters by step_size times the unit direction."""
    params = list(params)
    if direction.numel() != sum(p.numel() for p in params):
        raise ValueError('Direction length does not match the selected parameters')
    offset = 0
    for p in params:
        size = p.numel()
        p.add_(step_size * direction[offset:offset + size].view_as(p).to(p))
        offset += size

In [ ]:
# Choose a shared parameter subspace and a FIXED training batch for all geometry checks.
def selected_parameters(model):
    return list(model.layer4[-1].parameters()) + list(model.fc.parameters())


tangent_loader = DataLoader(stack_train, batch_size=TANGENT_BATCH_SIZE, shuffle=False)
calibration_batch = next(iter(tangent_loader))
tangents = []
for digit_a, digit_b in PARETO_BETA_PAIRS:
    beta = torch.zeros(10)
    beta[digit_a], beta[digit_b] = 1.0, -1.0
    result = minres_teacher_tangent(
        soft_seed, calibration_batch, beta, params=selected_parameters(soft_seed),
        max_iter=MINRES_MAX_ITER, tolerance=MINRES_TOLERANCE)
    result['direction'] = result['direction'].cpu()
    tangents.append((digit_a, digit_b, result))
    print(f'beta=e{digit_a}-e{digit_b} | alpha={result["alpha"]}')
    print('  calibration stationarity ratio:', result['stationarity_ratio'],
          '| FW gap:', result['fw_gap'],
          '| MINRES relative residual:', result['minres_relative_residual'])
    print('  predicted loss slopes for a positive unit step:', result['predicted_slopes'])
    if not result['minres_converged']:
        print('  Solve is truncated: increase MINRES_MAX_ITER to check direction stability.')
    if result['stationarity_ratio'] > 0.1:
        print('  Calibration stationarity ratio is high; inspect training and batch noise.')

# Inspect whether the ten task gradients actually conflict on this calibration batch.
if tangents:
    fig, ax = plt.subplots(figsize=(5, 4))
    heatmap = ax.imshow(tangents[0][2]['gradient_cosines'], vmin=-1, vmax=1, cmap='coolwarm')
    ax.set_xticks(range(10))
    ax.set_yticks(range(10))
    ax.set_xlabel('Digit objective')
    ax.set_ylabel('Digit objective')
    ax.set_title('Gradient cosine: negative means local conflict')
    fig.colorbar(heatmap, ax=ax)
    plt.show()

## 13. Correct each candidate and compare all ten validation losses

The predictor moves parameters by `theta_new = theta_seed + sign * step * v`.
The corrector then minimizes a positive weighted sum of the ten soft losses.
It updates all student parameters with BatchNorm statistics fixed. Thus the
**predictor** uses a parameter subspace, while the **corrector** can move the
whole model. All candidates start from the same seed and use the same
training-batch order for correction.

The sign of the correction preference matters. Differentiating the stationary
condition with weights gives `H delta_theta + Gᵀ delta_alpha ≈ 0`. Therefore a
positive predictor step satisfying `H v = Gᵀ beta` corresponds locally to
a **negative** change in task weights along `beta`. We use the heuristic
`softmax(log(alpha + floor) - sign * strength * beta)` to tilt weights with
that sign while keeping them positive. It is not an exact continuation of
optimal weights. The observed losses decide whether the candidate is useful.

Before correction, we compare predicted and observed loss changes on the
calibration batch. After correction, we record its stationarity diagnostic
and evaluate on validation pairs. One epoch is a demonstration budget, not a
guarantee of convergence. Teacher BCE measures imitation; exact match and F1
measure agreement with the true digit labels.

A dominates B if its teacher loss is no higher for **every** digit and strictly
lower for **at least one**. We keep points that no other sampled point dominates.
For example, losses `(0.2, 0.5)` and `(0.3, 0.4)` trade off, while `(0.4, 0.6)`
is dominated by both. The actual comparison uses all ten coordinates.

In [ ]:
# Store independent CPU snapshots so later training cannot mutate a saved state.
def cpu_state_dict(model):
    return {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}


def correction_preference(alpha, beta, sign, strength=PREFERENCE_STRENGTH):
    """Positive weights with a tilt opposite the predictor's beta sign."""
    alpha = torch.as_tensor(alpha, dtype=torch.float32)
    beta = torch.as_tensor(beta, dtype=alpha.dtype, device=alpha.device)
    if sign not in (-1, 1) or strength < 0:
        raise ValueError('sign must be -1 or +1 and strength must be nonnegative')
    return torch.softmax(alpha.clamp_min(1e-4).log() - sign * strength * beta, dim=0)


def correct_candidate(model, preference, epochs=CORRECTOR_EPOCHS):
    # Reset the batch-order generator for each candidate, reducing comparison noise.
    loader = DataLoader(stack_train, batch_size=BATCH_SIZE, shuffle=True,
                        generator=torch.Generator().manual_seed(SEED + 100))
    weights = preference.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    model.eval()  # gradients stay enabled; freeze the same BatchNorm buffers as the tangent
    for epoch in range(epochs):
        total_loss = total_samples = 0
        for images, _, _, img_a, img_b in loader:
            teacher = soft_teacher_targets(img_a, img_b)
            logits = model(images.to(DEVICE))
            losses = per_digit_distillation_losses(logits, teacher)
            loss = (weights * losses).sum()  # one scalar to backpropagate, retaining task priorities
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * images.size(0)
            total_samples += images.size(0)
        print(f'  corrector epoch {epoch + 1}/{epochs}: {total_loss / total_samples:.4f}')
    return model


@torch.no_grad()
def calibration_losses(model, batch):
    # Observe the same losses whose local derivatives predicted the tangent effect.
    model.eval()
    images, _, _, img_a, img_b = batch
    return per_digit_distillation_losses(
        model(images.to(DEVICE)), soft_teacher_targets(img_a, img_b)).cpu()


@torch.no_grad()
def evaluate_front_point(model, loader):
    model.eval()
    total = exact = 0
    teacher_loss_sum = torch.zeros(10, device=DEVICE)
    tp = torch.zeros(10, device=DEVICE)
    fp = torch.zeros(10, device=DEVICE)
    fn = torch.zeros(10, device=DEVICE)
    for images, targets, _, img_a, img_b in loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        teacher = soft_teacher_targets(img_a, img_b)
        logits = model(images)
        # Sum per sample, then divide by TOTAL samples after all batches.
        per_sample = F.binary_cross_entropy_with_logits(logits, teacher, reduction='none')
        teacher_loss_sum += per_sample.sum(dim=0)
        predicted = prediction_flags(torch.sigmoid(logits), rule='fixed')
        exact += (predicted == targets).all(dim=1).sum().item()
        # F1 balances false alarms (FP) and missed digits (FN), ignoring true negatives.
        tp += (predicted * targets).sum(dim=0)
        fp += (predicted * (1 - targets)).sum(dim=0)
        fn += ((1 - predicted) * targets).sum(dim=0)
        total += images.size(0)
    if total == 0:
        raise ValueError('Evaluation loader is empty')
    f1 = 2 * tp / (2 * tp + fp + fn).clamp_min(1)
    return {
        'teacher_losses': (teacher_loss_sum / total).cpu().tolist(),
        'exact_match': exact / total,
        'f1_per_digit': f1.cpu().tolist(),
        'f1_mean': f1.mean().item(),
    }

In [ ]:
# Keep all sampled points for which no other point is better in the Pareto sense.
def pareto_front(points, maximize=True):
    """Return non-dominated indices; use maximize=False for the ten teacher losses."""
    if len(points) == 0:
        return []
    P = torch.tensor([list(map(float, p)) for p in points], dtype=torch.float64)
    if P.ndim != 2 or P.shape[1] == 0 or not torch.isfinite(P).all():
        raise ValueError('Objectives must be a nonempty, finite matrix')
    # Convert minimization to maximization so the same comparison handles both cases.
    if not maximize:
        P = -P
    keep = []
    for i in range(len(P)):
        # Equal points do not dominate each other: at least one strict improvement is required.
        dominated = any(bool((P[j] >= P[i]).all() and (P[j] > P[i]).any())
                        for j in range(len(P)) if j != i)
        if not dominated:
            keep.append(i)
    return keep


# A known trade-off and one inferior point check the minimization convention.
assert pareto_front([[0.2, 0.5], [0.3, 0.4], [0.4, 0.6]], maximize=False) == [0, 1]

In [ ]:
# Save a common seed and include it in comparisons: a new candidate must earn its place.
seed_path = ARTIFACT_DIR / 'generalist_soft_seed.pt'
torch.save(cpu_state_dict(soft_seed), seed_path)
seed_metrics = evaluate_front_point(soft_seed, val_loader)
seed_diagnostics = measure_stationarity(soft_seed, calibration_batch, selected_parameters(soft_seed))
candidate_results = [dict(name='soft_seed', checkpoint=str(seed_path), kind='seed',
                          beta=None, sign=0, step_size=0.0,
                          stationarity=seed_diagnostics, **seed_metrics)]

# The direction has unit length; scale its distance relative to selected parameter magnitude.
selected_norm = sqrt(sum(p.detach().float().square().sum().item()
                         for p in selected_parameters(soft_seed)))
step_size = RELATIVE_TANGENT_STEP * selected_norm
print('Selected-parameter norm:', selected_norm, '| tangent step:', step_size)
soft_seed.cpu()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

# Every child reloads the SAME seed, so steps do not accidentally accumulate across candidates.
for digit_a, digit_b, tangent in tangents:
    for sign in (1, -1):
        name = f'tangent_{digit_a}_{digit_b}_{"plus" if sign > 0 else "minus"}'
        candidate = make_resnet50(num_outputs=10).to(DEVICE)
        candidate.load_state_dict(torch.load(seed_path, map_location='cpu', weights_only=True))
        apply_tangent_step(selected_parameters(candidate), tangent['direction'], sign * step_size)

        # First-order prediction can fail for a large step: compare it with an actual forward pass.
        predicted_change = sign * step_size * np.asarray(tangent['predicted_slopes'])
        observed_change = calibration_losses(candidate, calibration_batch) - tangent['teacher_losses']
        print(name, '| predicted calibration loss change:', predicted_change.round(5))
        print('  observed calibration loss change:', observed_change.numpy().round(5))

        preference = correction_preference(tangent['alpha'], tangent['beta'], sign)
        candidate = correct_candidate(candidate, preference)
        diagnostics = measure_stationarity(candidate, calibration_batch, selected_parameters(candidate))
        metrics = evaluate_front_point(candidate, val_loader)
        checkpoint = ARTIFACT_DIR / f'{name}.pt'
        torch.save(cpu_state_dict(candidate), checkpoint)
        candidate_results.append(dict(
            name=name, checkpoint=str(checkpoint), kind='corrected_candidate',
            beta=[digit_a, digit_b], sign=sign, step_size=sign * step_size,
            preference=preference.tolist(), stationarity=diagnostics,
            minres_residual=tangent['minres_relative_residual'],
            predicted_calibration_change=predicted_change.tolist(),
            observed_calibration_change=observed_change.tolist(), **metrics))
        print('  validation exact match:', round(metrics['exact_match'], 4),
              '| mean teacher BCE:', round(np.mean(metrics['teacher_losses']), 4),
              '| calibration stationarity ratio:', round(diagnostics['stationarity_ratio'], 4))
        del candidate
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

# All ten loss coordinates determine dominance, even when the later figure shows only two.
front_indices = pareto_front([row['teacher_losses'] for row in candidate_results], maximize=False)
for index, row in enumerate(candidate_results):
    row['nondominated'] = index in front_indices
    print(row['name'], '| nondominated among endpoints:', row['nondominated'])

## 14. Sample continuous lines between the seed and its children

For any `t` in `[0,1]`, form `theta(t) = (1-t) theta_seed + t theta_child`.
The weights vary continuously, but the losses need not follow a straight line
and the predicted digit set can change abruptly at a threshold. This run
samples the fractions in `INTERPOLATION_FRACTIONS` and saves each model.

These are lines within a local parameter approximation. With ten objectives,
they do not cover a whole Pareto surface. We evaluate each sampled model and
recompute dominance over **endpoints and interpolated samples together**.
Fixed BatchNorm buffers are shared across the seed and corrected children.
The interpolation function also handles floating buffers and integer counters.

In [ ]:
# Interpolate entire saved states so each fraction can be reconstructed and evaluated.

def interpolate_state(parent_state, child_state, fraction):
    if not 0.0 <= fraction <= 1.0:
        raise ValueError('fraction must be in [0, 1]')
    result = {}
    for key, parent_value in parent_state.items():
        child_value = child_state[key]
        # Float parameters are interpolated; integer counters must remain integers.
        if parent_value.is_floating_point():
            result[key] = torch.lerp(parent_value, child_value, fraction)
        else:
            result[key] = parent_value.clone() if fraction < 0.5 else child_value.clone()
    return result


parent_state = torch.load(seed_path, map_location='cpu', weights_only=True)
interpolated_results = []
for row in candidate_results[1:]:
    if not row['nondominated']:
        continue
    child_state = torch.load(row['checkpoint'], map_location='cpu', weights_only=True)
    for fraction_index, fraction in enumerate(INTERPOLATION_FRACTIONS):
        local_model = make_resnet50(num_outputs=10).to(DEVICE)
        local_model.load_state_dict(interpolate_state(parent_state, child_state, fraction))
        metrics = evaluate_front_point(local_model, val_loader)
        name = f'interp_{row["name"]}_{fraction_index}'
        checkpoint = ARTIFACT_DIR / f'{name}.pt'
        torch.save(cpu_state_dict(local_model), checkpoint)
        interpolated_results.append(dict(name=name, checkpoint=str(checkpoint),
                                         kind='interpolated_sample', parent='soft_seed',
                                         child=row['name'], fraction=fraction, **metrics))
        print('interpolate', row['name'], 't=', fraction,
              '| exact=', round(metrics['exact_match'], 4))
        del local_model
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

# An interpolation may dominate an endpoint: refresh the front using the complete sample.
all_points = candidate_results + interpolated_results
sampled_front_indices = pareto_front([row['teacher_losses'] for row in all_points],
                                     maximize=False)
for index, row in enumerate(all_points):
    row['sampled_nondominated'] = index in sampled_front_indices

fig, ax = plt.subplots(figsize=(6, 5))
for row in candidate_results:
    ax.scatter(row['teacher_losses'][7], row['teacher_losses'][9],
               marker='o', color='tab:blue' if row['sampled_nondominated'] else 'lightgray')
    ax.annotate(row['name'], (row['teacher_losses'][7], row['teacher_losses'][9]), fontsize=7)
for row in interpolated_results:
    ax.scatter(row['teacher_losses'][7], row['teacher_losses'][9],
               marker='x', color='tab:orange' if row['sampled_nondominated'] else 'lightgray')
ax.set_xlabel('Validation teacher BCE for digit 7 (lower is better)')
ax.set_ylabel('Validation teacher BCE for digit 9 (lower is better)')
ax.set_title('Ten-loss dominance projected onto digits 7 and 9')
print('Circles: endpoints; crosses: interpolation; gray: dominated in the sampled set.')
plt.show()

## 15. Select on validation data, then test once

Choose the model with the highest validation exact match among **all sampled
non-dominated points**, including saved interpolations. The teacher-loss
trade-offs filter the eligible set; true-label recognition chooses one model
within it. This is a declared selection rule, not a claim that one Pareto
point is universally best. Evaluate that choice once on the final test set.

In [ ]:
# Use the final sampled front consistently, including any interpolation that beats an endpoint.

eligible = [row for row in all_points if row['sampled_nondominated']]
chosen = max(eligible, key=lambda row: row['exact_match'])
final_model = make_resnet50(num_outputs=10).to(DEVICE)
final_model.load_state_dict(torch.load(chosen['checkpoint'], map_location='cpu',
                                       weights_only=True))
# This is the first test-set evaluation in the notebook.
final_metrics = evaluate_front_point(final_model, test_loader)
print('Selected by validation exact match:', chosen['name'])
print('Final test exact match:', final_metrics['exact_match'])
print('Final test mean per-digit F1:', final_metrics['f1_mean'])

summary = {
    'settings': {
        'seed': SEED, 'batch_size': BATCH_SIZE, 'specialist_epochs': SPEC_EPOCHS,
        'specialist_train_size': SPEC_TRAIN_SIZE, 'baseline_epochs': GEN_EPOCHS,
        'soft_seed_epochs': SOFT_SEED_EPOCHS, 'batchnorm_frozen_for_exploration': True,
        'stack_train_samples': STACK_TRAIN_SAMPLES, 'stack_val_samples': STACK_VAL_SAMPLES,
        'stack_test_samples': STACK_TEST_SAMPLES, 'beta_pairs': PARETO_BETA_PAIRS,
        'tangent_batch_size': TANGENT_BATCH_SIZE, 'minres_tolerance': MINRES_TOLERANCE,
        'preference_strength': PREFERENCE_STRENGTH,
        'interpolation_fractions': INTERPOLATION_FRACTIONS,
        'corrector_epochs': CORRECTOR_EPOCHS,
        'minres_max_iter': MINRES_MAX_ITER,
        'relative_tangent_step': RELATIVE_TANGENT_STEP,
        'parameter_subspace': 'last ResNet50 layer4 block plus fc',
    },
    'tangent_diagnostics': [dict(beta=result['beta'].tolist(), alpha=result['alpha'],
                                stationarity_ratio=result['stationarity_ratio'],
                                fw_gap=result['fw_gap'], minres_iterations=result['minres_iterations'],
                                minres_residual=result['minres_relative_residual'],
                                minres_converged=result['minres_converged'],
                                predicted_slopes=result['predicted_slopes'])
                            for _, _, result in tangents],
    'seed_and_corrected_candidates': candidate_results,
    'interpolated_samples': interpolated_results,
    'selected_name': chosen['name'],
    'selected_checkpoint': chosen['checkpoint'],
    'final_test': final_metrics,
}
with (ARTIFACT_DIR / 'pareto_summary.json').open('w', encoding='utf-8') as handle:
    json.dump(summary, handle, indent=2, allow_nan=False)
print('Saved:', ARTIFACT_DIR / 'pareto_summary.json')

## Read the results in this order

| Output | What it tells you | What to check |
|---|---|---|
| Specialist recall | Whether each teacher recognizes its own digit | Weak teachers give weak teaching signals. |
| Gradient cosine matrix | Whether task gradients disagree locally | Negative entries indicate immediate conflict; zero-norm gradients need separate care. |
| `alpha` and `fw_gap` | The gradient balance and accuracy of the weight solve | `alpha` sums to one; the gap should be small for the scaled Gram objective. |
| `stationarity_ratio` | Size of the remaining balanced gradient | Applies only to the selected parameters and calibration batch. |
| `minres_relative_residual` | How closely the linear system was solved | Small means an accurate linear solve, not an accurate digit classifier. |
| Predicted vs observed calibration change | Whether the local tangent prediction fits the chosen step | Large disagreement suggests checking step size and solve accuracy. |
| Ten validation teacher losses | How well a model follows each specialist | These ten values determine sampled dominance. |
| Validation exact match / F1 | Whether the predicted digit sets are correct | These use true labels; teacher imitation can disagree with recognition quality. |
| Final test metrics | Performance of the already-selected model | Do not change settings using these results and still call the test untouched. |

A useful experiment reports trade-offs and diagnostics together. A small
set of non-dominated samples supports a local exploratory result; it does
not establish the complete ten-dimensional Pareto front.